# CyberSOC Arena - GRPO training on Qwen2.5-0.5B-Instruct

This notebook trains a small LLM to play **CyberSOC Arena**, an OpenEnv environment where the model acts as a Tier-2 SOC analyst, using Hugging Face TRL's `GRPOTrainer` with the live environment as the reward function.

**Runs on a free Colab T4** (about 25 minutes for 80 GRPO steps with 4 generations per prompt).

Outputs:
- `runs/colab_grpo/training_log.json` - per-step loss, reward, completion length
- `runs/colab_grpo/reward_curve.png`  - learning curve
- `runs/colab_grpo/loss_curve.png`    - GRPO loss curve

Repo: <https://github.com/amit51/cybersoc-arena>  -  HF Space: <https://huggingface.co/spaces/amit51/cybersoc-arena>

## 1. Install + clone

TRL >=0.18 has built-in OpenEnv support; Unsloth gives us 2x faster training and 70% less VRAM.

In [ ]:
!pip install -q --upgrade openenv-core trl transformers accelerate peft bitsandbytes
!pip install -q --upgrade unsloth || echo 'unsloth optional - falling back to pure TRL'
!pip install -q matplotlib pydantic

In [ ]:
# Pull the env package straight from the HF Space (also pip-installable)
!pip install -q git+https://huggingface.co/spaces/amit51/cybersoc-arena || git clone https://github.com/amit51/cybersoc-arena.git && cd cybersoc-arena && pip install -e .

## 2. Sanity-check the env

In [ ]:
from cybersoc_arena import CyberSOCEnv, CyberAction, CurriculumEnv, SCENARIO_TYPES, TIERS
print('Scenarios:', SCENARIO_TYPES)
print('Curriculum tiers:', [(t.index, t.name, t.advance_threshold) for t in TIERS])

env = CyberSOCEnv()
obs = env.reset(seed=42, scenario_type='multi_stage_chain')
print('\nALERT:', obs.alert.summary[:120])
print('Hosts:', obs.asset_inventory.hosts)
print('Visible IPs:', obs.asset_inventory.visible_ips)
print('Step budget:', obs.step_budget)
obs = env.step(CyberAction(action_type='query_logs', ip=obs.asset_inventory.visible_ips[0]))
print('\nAfter one step: reward =', obs.reward, ' evidence =', obs.evidence_count)

## 3. Set up GRPO

We frame each *step* of CyberSOC Arena as a single GRPO prompt: the prompt = the current observation rendered as text + the JSON action schema. The completion = a single JSON action object. The reward function resets a fresh env, replays the agent's history (so episodes stay deterministic), and returns the per-step reward plus a small evidence-quality bonus.

In [ ]:
import json, re, random
from cybersoc_arena import CyberSOCEnv, CyberAction
from cybersoc_arena.actions import parse_action

def render_obs_as_prompt(obs) -> str:
    lines = [
        f'You are a Tier-2 SOC analyst. Investigate the alert and decide.',
        f'',
        f'ALERT [{obs.alert.severity.upper()}]: {obs.alert.summary}',
        f'Step {obs.step}/{obs.step_budget}  (remaining: {obs.remaining_steps})',
        f'Hosts: {obs.asset_inventory.hosts}',
        f'Visible IPs: {obs.asset_inventory.visible_ips}',
        f'',
        f'Evidence so far ({obs.evidence_count} items):',
    ]
    for e in obs.evidence_collected[-6:]:
        lines.append(f'  - [{e.action}({e.target})] {e.finding[:140]}')
    lines += [
        f'',
        f'Available actions: {obs.available_actions}',
        f'',
        f'Reply with ONE JSON action, e.g. {{"action_type": "query_logs", "ip": "10.0.0.5"}}',
    ]
    return '\n'.join(lines)

def env_reward_function(prompts, completions, **kwargs):
    rewards = []
    for completion in completions:
        text = completion if isinstance(completion, str) else completion[0]['content']
        try:
            act = parse_action(text)
            ca = CyberAction(action_type=act.action_type, ip=act.ip,
                              host=act.host, entity=act.entity, summary=act.summary)
        except Exception:
            rewards.append(-0.10)  # malformed action penalty
            continue
        env = CyberSOCEnv()
        # Replay the prompt's seed (encoded in the prompt's metadata).
        # In this minimal demo we just sample a fresh scenario each time.
        obs = env.reset(seed=random.randint(0, 1_000_000),
                         scenario_type=random.choice(SCENARIO_TYPES))
        try:
            obs = env.step(ca)
            rewards.append(float(obs.reward))
        except Exception as e:
            rewards.append(-0.20)
    return rewards

In [ ]:
import os
from datasets import Dataset

# Build a small prompt dataset: one prompt per starting observation.
def build_prompt_dataset(n_prompts=80, seed=0):
    random.seed(seed)
    prompts = []
    for k in range(n_prompts):
        env = CyberSOCEnv()
        scen = random.choice(SCENARIO_TYPES)
        obs = env.reset(seed=seed + k, scenario_type=scen)
        prompts.append({'prompt': render_obs_as_prompt(obs), 'scenario': scen})
    return Dataset.from_list(prompts)

train_ds = build_prompt_dataset(n_prompts=80, seed=42)
print(train_ds, '\n')
print(train_ds[0]['prompt'][:600])

## 4. Train (TRL GRPOTrainer + Qwen2.5-0.5B-Instruct + LoRA)

On a free T4, 80 prompts x 4 generations should finish in roughly 25 minutes. Bump `num_train_epochs` if you have credits to burn.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.bfloat16, device_map='auto',
)

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)

config = GRPOConfig(
    output_dir='runs/colab_grpo',
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_prompt_length=768,
    max_completion_length=128,
    learning_rate=5e-6,
    logging_steps=2,
    report_to='none',
    bf16=True,
)
trainer = GRPOTrainer(
    model=model,
    processing_class=tok,
    reward_funcs=env_reward_function,
    args=config,
    train_dataset=train_ds,
    peft_config=peft_config,
)
trainer.train()

## 5. Plot the curves

In [ ]:
import json, os, matplotlib.pyplot as plt, numpy as np
log = trainer.state.log_history
steps = [r['step'] for r in log if 'loss' in r]
losses = [r['loss'] for r in log if 'loss' in r]
rewards = [r.get('reward', float('nan')) for r in log if 'reward' in r]
rsteps = [r['step'] for r in log if 'reward' in r]

os.makedirs('runs/colab_grpo', exist_ok=True)
with open('runs/colab_grpo/training_log.json', 'w') as f:
    json.dump(log, f, indent=2)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(steps, losses, label='GRPO loss', color='#7570b3')
ax.set_xlabel('Training step (#)'); ax.set_ylabel('Loss'); ax.set_title('CyberSOC Arena - GRPO loss curve')
ax.grid(alpha=0.3); ax.legend(); fig.tight_layout(); fig.savefig('runs/colab_grpo/loss_curve.png', dpi=130)

if rewards:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(rsteps, rewards, label='mean reward', color='#1b9e77')
    ax.set_xlabel('Training step (#)'); ax.set_ylabel('Mean per-step env reward')
    ax.set_title('CyberSOC Arena - GRPO reward curve')
    ax.grid(alpha=0.3); ax.legend(); fig.tight_layout()
    fig.savefig('runs/colab_grpo/reward_curve.png', dpi=130)
print('Saved runs/colab_grpo/loss_curve.png and runs/colab_grpo/reward_curve.png')

## 6. Eval: trained vs untrained on a held-out scenario

Drop a 0.5B base policy into 30 fresh episodes; compare cumulative reward and success rate against the trained adapter.

In [ ]:
import random
def rollout(model_for_inference, tok, n=30, seed=999):
    rs = []
    correct = 0
    for k in range(n):
        scen = random.choice(SCENARIO_TYPES)
        env = CyberSOCEnv()
        obs = env.reset(seed=seed + k, scenario_type=scen)
        total = 0.0
        while not obs.done:
            prompt = render_obs_as_prompt(obs)
            ids = tok(prompt, return_tensors='pt').to(model_for_inference.device)
            out = model_for_inference.generate(**ids, max_new_tokens=80, do_sample=False)
            text = tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True)
            try:
                act = parse_action(text)
                ca = CyberAction(action_type=act.action_type, ip=act.ip,
                                  host=act.host, entity=act.entity, summary=act.summary)
                obs = env.step(ca)
            except Exception:
                obs = env.step(CyberAction(action_type='query_logs',
                                            ip=obs.asset_inventory.visible_ips[0]))
            total += obs.reward
        if env.state.terminal_correct:
            correct += 1
        rs.append(total)
    return rs, correct / n

# Trained
trained_rewards, trained_acc = rollout(trainer.model, tok)
print(f'Trained:   mean={sum(trained_rewards)/len(trained_rewards):+.3f}  success={trained_acc:.1%}')

# Reload untrained base model for comparison
base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map='auto')
untrained_rewards, untrained_acc = rollout(base, tok)
print(f'Untrained: mean={sum(untrained_rewards)/len(untrained_rewards):+.3f}  success={untrained_acc:.1%}')

## 7. Push everything to HF Hub (so the Colab T4 results survive disconnect)

This pushes the LoRA adapter, the per-step `training_log.json`, the 2 PNG plots, and an auto-generated model card to a NEW model repo `amit51/cybersoc-arena-qwen0.5b-grpo`. That keeps the Colab T4 results separate from the A100 run (which uses `amit51/cybersoc-arena-qwen2.5-1.5b-grpo`).


In [ ]:
# Push the trained adapter, training_log.json, and the 2 PNGs to a model repo.
# Doing it this way means the artifacts survive even if Colab disconnects.

from huggingface_hub import HfApi, login, create_repo
import os, json

REPO = 'amit51/cybersoc-arena-qwen0.5b-grpo'  # change this if you want a different name

login()  # paste your HF token (Write scope)
create_repo(REPO, repo_type='model', exist_ok=True)

# 1. LoRA adapter + tokenizer
trainer.model.push_to_hub(REPO)
tok.push_to_hub(REPO)

# 2. training_log.json + the two PNGs from cell 12
api = HfApi()
for fname in ('training_log.json', 'loss_curve.png', 'reward_curve.png'):
    p = f'runs/colab_grpo/{fname}'
    if os.path.exists(p):
        api.upload_file(
            path_or_fileobj=p,
            path_in_repo=fname,
            repo_id=REPO,
            repo_type='model',
            commit_message=f'Add {fname} from Colab T4 GRPO run',
        )
        print(f'pushed {fname}')

# 3. minimal model card so the page is presentable
MD = f'''---
base_model: {MODEL}
library_name: peft
tags:
  - openenv
  - cybersoc-arena
  - grpo
  - cybersecurity
license: apache-2.0
---

# CyberSOC Arena GRPO ({MODEL})

LoRA adapter trained with `trl.GRPOTrainer` on a free Colab T4 against the
[CyberSOC Arena](https://huggingface.co/spaces/amit51/cybersoc-arena) OpenEnv.

## Eval (greedy rollout, n=30 mixed scenarios)

| Agent | Mean episode reward | Success rate |
|---|---:|---:|
| Untrained {MODEL} | {sum(untrained_rewards)/len(untrained_rewards):+.3f} | {untrained_acc:.1%} |
| **GRPO-trained**  | **{sum(trained_rewards)/len(trained_rewards):+.3f}** | **{trained_acc:.1%}** |

![loss curve](loss_curve.png)
![reward curve](reward_curve.png)
'''
api.upload_file(
    path_or_fileobj=MD.encode('utf-8'),
    path_in_repo='README.md',
    repo_id=REPO,
    repo_type='model',
    commit_message='Add model card with eval table + plots',
)
print(f'Done. View at https://huggingface.co/{REPO}')
